# W10

(2bp) **1. Update the cross validation example to accept CNN models. Combine the top 3 models (classification on Fashion-MNIST) into an ensemble and analyze if the ensemble’s performance is better than that of the individual models.**

In [ ]:
!pip install pyefd

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sn
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

def prepare_data():
    (X_train, Y_train), (X_test, Y_test) = fashion_mnist.load_data()

    # Normalize pixel values to range [0, 1]
    X_train = X_train / 255.0
    X_test = X_test / 255.0

    return X_train, Y_train, X_test, Y_test

# Reshape the input data for CNN models
X_train, Y_train, X_test, Y_test = prepare_data()
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1)
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1)

# Define the model architecture
def create_model():
    model = Sequential([
        Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
        MaxPooling2D(pool_size=(2, 2)),
        Flatten(),
        Dense(64, activation='relu'),
        Dense(10)
    ])
    return model

# Perform cross-validation
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
ensemble_predictions = np.zeros((len(X_test), 10))
individual_predictions = []

for train_index, val_index in skf.split(X_train, Y_train):
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    Y_train_fold, Y_val_fold = Y_train[train_index], Y_train[val_index]

    models = []
    top_models = []
    top_models_acc = []

    # Train individual models and track the top models based on validation accuracy
    for i in range(3):  # Create 3 models for ensemble
        model = create_model()
        model.compile(optimizer='adam',
                      loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                      metrics=['accuracy'])
        model.fit(X_train_fold, Y_train_fold, epochs=15, verbose=0)

        preds_val = model.predict(X_val_fold)
        predictions_val = np.argmax(preds_val, axis=1)
        acc_val = accuracy_score(Y_val_fold, predictions_val)

        if len(top_models) < 3:
            top_models.append(model)
            top_models_acc.append(acc_val)
        else:
            min_acc_index = top_models_acc.index(min(top_models_acc))
            if acc_val > top_models_acc[min_acc_index]:
                top_models[min_acc_index] = model
                top_models_acc[min_acc_index] = acc_val

    top_models = sorted(top_models, key=lambda x: accuracy_score(Y_test, np.argmax(x.predict(X_test), axis=1)), reverse=True)
    
    # Ensemble model predictions
    fold_individual_predictions = []
    for model in top_models:
        preds_test = model.predict(X_test)
        ensemble_predictions += np.argmax(preds_test, axis=1).reshape(-1, 1)
        preds_val = model.predict(X_val_fold)
        fold_individual_predictions.append(np.argmax(preds_val, axis=1))

    individual_predictions.extend(fold_individual_predictions)
    ensemble_predictions /= n_splits

# Calculate ensemble accuracy on the test set
ensemble_predictions = np.argmax(ensemble_predictions, axis=1)
ensemble_acc = accuracy_score(Y_test, ensemble_predictions)
print("Ensemble Classification accuracy on test set: {:.4f}".format(ensemble_acc))

# Calculate accuracy for each individual model on the test set
individual_predictions = np.array(individual_predictions)
individual_predictions = individual_predictions.T

individual_acc = []
for i, model in enumerate(top_models):
    preds_test = model.predict(X_test)
    individual_pred = np.argmax(preds_test, axis=1)
    acc = accuracy_score(Y_test, individual_pred)
    individual_acc.append(acc)
    print("Model {} accuracy on test set: {:.4f}".format(i+1, acc))


375/375 [==============================] - 1s 4ms/step
Ensemble Classification accuracy on test set: 0.1000
313/313 [==============================] - 1s 4ms/step
Model 1 accuracy on test set: 0.9117
313/313 [==============================] - 1s 4ms/step
Model 2 accuracy on test set: 0.9095
313/313 [==============================] - 1s 4ms/step
Model 3 accuracy on test set: 0.9058


In [ ]:
# Reshape individual_predictions to match the expected shape
#individual_predictions = individual_predictions.reshape(len(X_test), -1)

# Create a DataFrame to visualize the individual models' predictions
#columns = ['Model {}'.format(i+1) for i in range(len(top_models))]
#df_individual = pd.DataFrame(individual_predictions, columns=columns)

# Create a confusion matrix to analyze the ensemble predictions
#cm = confusion_matrix(Y_test, ensemble_predictions)
#df_cm = pd.DataFrame(cm, index=[i for i in range(10)], columns=[i for i in range(10)])

# Visualize the confusion matrix
#plt.figure(figsize=(10, 8))
#sn.heatmap(df_cm, annot=True, cmap='Blues')
#plt.xlabel('Predicted')
#plt.ylabel('True')
#plt.title('Confusion Matrix')
#plt.show()

(2bp) **2. Implement the K-Fold cross-validation to evaluate your CNN model over the Fashion-MNIST dataset. Keep the test data as your benchmark, using just the training data for architecture decisions. Compute the number of parameters, and compare to the result provided by Keras.**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.model_selection import KFold

# Load the Fashion-MNIST dataset
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

# Normalize pixel values to the range [0, 1]
X_train_full = X_train_full.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Add a channel dimension to the data
X_train_full = np.expand_dims(X_train_full, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

# Convert labels to one-hot encoded vectors
num_classes = 10
y_train_full = tf.keras.utils.to_categorical(y_train_full, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

# Split the training data into training and validation sets
val_ratio = 0.2
val_samples = int(val_ratio * len(X_train_full))

X_train = X_train_full[:-val_samples]
y_train = y_train_full[:-val_samples]
X_val = X_train_full[-val_samples:]
y_val = y_train_full[-val_samples:]

# CNN model architecture
model = Sequential()
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))

# Compiling the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Performing K-Fold cross-validation on the training data
k = 5  # Number of folds
kf = KFold(n_splits=k, shuffle=True)

fold_accs = []  # Store accuracy for each fold

for train_index, val_index in kf.split(X_train):
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

    # Training the model on the current fold
    model.fit(X_train_fold, y_train_fold, batch_size=64, epochs=10, verbose=1)

    # Evaluating the model on the validation set of the current fold
    _, accuracy = model.evaluate(X_val_fold, y_val_fold, verbose=0)
    fold_accs.append(accuracy)

# Calculating the mean and standard deviation of the fold accuracies
mean_acc = np.mean(fold_accs)
std_acc = np.std(fold_accs)

print(f"Mean accuracy: {mean_acc:.4f}")
print(f"Standard deviation: {std_acc:.4f}")

# Evaluate the model on the test set
_, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.4f}")

# Count the number of parameters in the model
num_params = model.count_params()
print(f"Number of parameters: {num_params}")
model.summary()

Epoch 1/10
600/600 [==============================] - 30s 49ms/step - loss: 0.5440 - accuracy: 0.8053
Epoch 2/10
600/600 [==============================] - 28s 46ms/step - loss: 0.3530 - accuracy: 0.8727
Epoch 3/10
600/600 [==============================] - 26s 44ms/step - loss: 0.3020 - accuracy: 0.8883
Epoch 4/10
600/600 [==============================] - 26s 44ms/step - loss: 0.2699 - accuracy: 0.9003
Epoch 5/10
600/600 [==============================] - 26s 44ms/step - loss: 0.2436 - accuracy: 0.9099
Epoch 6/10
600/600 [==============================] - 27s 45ms/step - loss: 0.2254 - accuracy: 0.9165
Epoch 7/10
600/600 [==============================] - 26s 44ms/step - loss: 0.2013 - accuracy: 0.9246
Epoch 8/10
600/600 [==============================] - 27s 44ms/step - loss: 0.1860 - accuracy: 0.9314
Epoch 9/10
600/600 [==============================] - 26s 44ms/step - loss: 0.1676 - accuracy: 0.9386
Epoch 10/10
600/600 [==============================] - 26s 44ms/step - loss: 0.153

(2bp) **3. Update the CNN autoencoder to have a latent with the same size as the Linear AE designed last week (37x1), using a sequence of Dense layers on the flatenned filter output. Compare its performance to the Linear AE and the SVC classifier.**

In [ ]:
import tensorflow as tf
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Load the Fashion-MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Normalize the pixel values
x_train = x_train / 255.0
x_test = x_test / 255.0

# Reshape input data to (samples, height, width, channels)
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1))
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1))

# Autoencoder network
encoder = tf.keras.Sequential([
    tf.keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same', input_shape=(28, 28, 1)),
    tf.keras.layers.MaxPooling2D((2, 2), padding='same'),
    tf.keras.layers.Conv2D(8, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2), padding='same'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(37, activation='relu')
])
decoder = tf.keras.Sequential([
    tf.keras.layers.Dense(392, activation='relu', input_shape=(37,)),
    tf.keras.layers.Reshape((7, 7, 8)),
    tf.keras.layers.Conv2DTranspose(8, (3, 3), strides=2, activation='relu', padding='same'),
    tf.keras.layers.Conv2DTranspose(16, (3, 3), strides=2, activation='relu', padding='same'),
    tf.keras.layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same'),
    tf.keras.layers.Reshape((28, 28))
])
autoencoder = tf.keras.Sequential([encoder, decoder])

# Compile the autoencoder model
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

# Train the autoencoder model
autoencoder.fit(x_train, x_train, epochs=10, validation_split=0.2)

#encoder = keras.models.Model(input_img, latent)

# Extract the features generated by the encoder of the autoencoder network
ae_features_model = tf.keras.Sequential(encoder.layers[:-1])
ae_features = ae_features_model.predict(x_train)

# Flatten the features for input to the classifier
ae_features_flat = ae_features.reshape(ae_features.shape[0], -1)

X_enc_train = encoder.predict(x_train)
X_enc_test = encoder.predict(x_test)

# Perform classification using SVC
scaler = StandardScaler()
X_enc_train = scaler.fit_transform(X_enc_train)
X_enc_test = scaler.transform(X_enc_test)
clf = SVC(C=1.5, kernel='rbf', gamma='auto')
clf.fit(X_enc_train, y_train)
predictions = clf.predict(X_enc_test)

# Evaluate classification accuracy
accuracy = accuracy_score(y_test, predictions)
print("Classification accuracy: {:.4f}".format(accuracy))

4422102/4422102 [==============================] - 0s 0us/step
Epoch 1/10
1500/1500 [==============================] - 104s 66ms/step - loss: 0.3318 - val_loss: 0.2978
Epoch 2/10
1500/1500 [==============================] - 87s 58ms/step - loss: 0.2917 - val_loss: 0.2902
Epoch 3/10
1500/1500 [==============================] - 84s 56ms/step - loss: 0.2871 - val_loss: 0.2871
Epoch 4/10
1500/1500 [==============================] - 90s 60ms/step - loss: 0.2845 - val_loss: 0.2849
Epoch 5/10
1500/1500 [==============================] - 89s 60ms/step - loss: 0.2830 - val_loss: 0.2836
Epoch 6/10
1500/1500 [==============================] - 90s 60ms/step - loss: 0.2818 - val_loss: 0.2830
Epoch 7/10
1500/1500 [==============================] - 85s 57ms/step - loss: 0.2809 - val_loss: 0.2818
Epoch 8/10
1500/1500 [==============================] - 90s 60ms/step - loss: 0.2802 - val_loss: 0.2811
Epoch 9/10
1500/1500 [==============================] - 85s 57ms/step - loss: 0.2796 - val_loss: 0.2806


(2bp) **4. Using strided convolution, implement a binary classifier for two of the MNIST classes, using Dense layers only where it is exclusively necessary (reduce the dimensions of the output to 1x1 output using strides).**

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist

# Load MNIST data
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

# Reshape images to 4D tensors with channel dimension (for convolutional layers)
train_images = train_images.reshape((60000, 28, 28, 1))
train_images = train_images.astype('float32') / 255

test_images = test_images.reshape((10000, 28, 28, 1))
test_images = test_images.astype('float32') / 255

# Create binary labels for digits 0 and 1
train_labels = ((train_labels == 0) | (train_labels == 1)).astype('float32')
test_labels = ((test_labels == 0) | (test_labels == 1)).astype('float32')

# Define the model
model = tf.keras.Sequential([
    # Convolutional layer with 32 filters, 3x3 kernel size, and stride of 2
    tf.keras.layers.Conv2D(32, (3, 3), strides=(2, 2), activation='relu', input_shape=(28, 28, 1)),
    # Convolutional layer with 64 filters, 3x3 kernel size, and stride of 2
    tf.keras.layers.Conv2D(64, (3, 3), strides=(2, 2), activation='relu'),
    # Flatten the output of the convolutional layers
    tf.keras.layers.Flatten(),
    # Dense layer with 64 units
    tf.keras.layers.Dense(64, activation='relu'),
    # Output layer with sigmoid activation (for binary classification)
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Train the model
model.fit(train_images, train_labels, epochs=5, batch_size=64, validation_split=0.1)

# Evaluate the model on the test data
test_loss, test_acc = model.evaluate(test_images, test_labels)
print('Test accuracy:', test_acc)

Epoch 1/5
844/844 [==============================] - 17s 19ms/step - loss: 0.0548 - accuracy: 0.9812 - val_loss: 0.0192 - val_accuracy: 0.9938
Epoch 2/5
844/844 [==============================] - 15s 18ms/step - loss: 0.0195 - accuracy: 0.9940 - val_loss: 0.0170 - val_accuracy: 0.9947
Epoch 3/5
844/844 [==============================] - 16s 19ms/step - loss: 0.0137 - accuracy: 0.9952 - val_loss: 0.0127 - val_accuracy: 0.9960
Epoch 4/5
844/844 [==============================] - 17s 21ms/step - loss: 0.0095 - accuracy: 0.9970 - val_loss: 0.0117 - val_accuracy: 0.9965
Epoch 5/5
313/313 [==============================] - 1s 4ms/step - loss: 0.0142 - accuracy: 0.9955
Test accuracy: 0.9955000281333923
